In [8]:
%%time

# Import libraries
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# Load dataset
df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)

# Prepare dataset

# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort dataset
df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# Next day closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker for which next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# Next close change
df_model['next_close_change'] = df_model['next_close'] - df_model['close']


# Next close change PCT - This will be the target
df_model['next_close_change_pct'] = (
    (df_model['next_close'] - df_model['close']) / df_model['close']
) * 100



# Target already created with dataset loading


# Remove final row per ticker for which next_close does not exist)
df_model = df_model[
    df_model['next_close'].notna()
].copy()



# Train/test split

# Train on 2021-2024 data
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test on 2025 data
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# Select features
exclude_cols = [
    'timestamp',
    'ticker',
    'next_close_change_pct'
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]


X_train = train_df[feature_cols]
y_train = train_df['next_close_change_pct']

X_test = test_df[feature_cols]
y_test = test_df['next_close_change_pct']


# Define model
model = xgb.XGBRegressor()


# Add GridSearch (not doing it in initial notebook, just saving my place)


# Train model
model.fit(X_train, y_train)


# Get predictinos
y_test_pred = model.predict(X_test)


# Define evaluation metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

# Print evaluation metrics
print("\n===== REGRESSION METRICS =====")

print(f"MAE:    {mae:.4f}")
print(f"MSE:    {mse:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MedAE:  {medae:.4f}")
print(f"R²:     {r2:.4f}")


===== REGRESSION METRICS =====
MAE:    0.1391
MSE:    0.7019
RMSE:   0.8378
MedAE:  0.0640
R²:     0.9320
CPU times: user 3.66 s, sys: 604 ms, total: 4.26 s
Wall time: 2.33 s


In [9]:
results_df = test_df.copy()

# Add predictions
results_df['pred_next_close_change_pct'] = (y_test_pred).round(2)

# Impute error
results_df['pred_error'] = results_df['pred_next_close_change_pct'] - results_df['next_close_change_pct']

# Absolute error
results_df['abs_pred_error'] = results_df['pred_error'].abs()

# Directionally correct
results_df['directionally_correct'] = (
    np.sign(results_df['next_close_change_pct']) ==
    np.sign(results_df['pred_next_close_change_pct'])
).astype(int)

results_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,close_lag1,close_lag2,return_lag1,next_close,next_close_change,next_close_change_pct,pred_next_close_change_pct,pred_error,abs_pred_error,directionally_correct
872,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,37.78,37.15,0.016958,35.71,-2.28,-6.001579,-5.91,0.091580,0.091580,1
873,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,37.99,37.78,0.005558,36.49,0.78,2.184262,1.86,-0.324262,0.324262,1
874,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,35.71,37.99,-0.060016,36.24,-0.25,-0.685119,-0.63,0.055119,0.055119,1
875,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,36.49,35.71,0.021843,36.00,-0.24,-0.662252,-0.63,0.032252,0.032252,1
876,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,36.24,36.49,-0.006851,35.91,-0.09,-0.250000,-0.23,0.020000,0.020000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478297,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,123.54,123.78,-0.001939,126.23,0.74,0.589688,0.61,0.020312,0.020312,1
478298,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,125.49,123.54,0.015784,125.98,-0.25,-0.198051,-0.16,0.038051,0.038051,1
478299,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,126.23,125.49,0.005897,126.41,0.43,0.341324,0.43,0.088676,0.088676,1
478300,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,125.98,126.23,-0.001981,125.82,-0.59,-0.466735,-0.44,0.026735,0.026735,1


In [10]:
results_df['directionally_correct'].value_counts()

directionally_correct
1    97793
0     1457
Name: count, dtype: int64

In [11]:
# Save
results_df.to_csv('models_daily_results/Round_1_XGBR.csv', index=False)